In [1]:
print(1)


1


In [2]:
import os
import sys
import findspark

# 1. KHAI BÁO BIẾN MÔI TRƯỜNG TRƯỚC
# Sử dụng Raw string (r'') là đúng, nhưng hãy kiểm tra kỹ dấu gạch chéo
spark_home = r'C:\Program Files\Spark\spark-3.5.8-bin-hadoop3'
java_home = r'C:\Program Files\Java\jdk-11'
hadoop_home = r'C:\Program Files\Winutils'

os.environ['SPARK_HOME'] = spark_home
os.environ['JAVA_HOME'] = java_home
os.environ['HADOOP_HOME'] = hadoop_home

# 2. KHỞI TẠO FINDSPARK (Truyền trực tiếp đường dẫn vào để chắc chắn)
findspark.init(spark_home)

# 3. SAU ĐÓ MỚI IMPORT CÁC THƯ VIỆN CỦA PYSPARK
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

try:
    # 4. KHỞI TẠO SESSION
    # Thêm cấu hình địa chỉ warehouse để tránh lỗi quyền ghi đè trên Windows
    spark = SparkSession.builder \
        .appName("OTTO_Recommendation_EDA") \
        .config("spark.driver.memory", "4g") \
        .config("spark.sql.warehouse.dir", "file:///C:/temp") \
        .getOrCreate()
    
    print("--- Chúc mừng! Spark đã khởi động thành công ---")
    print("Phiên bản Spark Core:", spark.version)
    
    # Test thử
    spark.createDataFrame([("Ngo Bo", "Pass")], ["Student", "Result"]).show()

except Exception as e:
    print("Vẫn còn lỗi. Hãy kiểm tra các mục sau:")
    print(f"Chi tiết lỗi: {e}")

--- Chúc mừng! Spark đã khởi động thành công ---
Phiên bản Spark Core: 3.5.8
+-------+------+
|Student|Result|
+-------+------+
| Ngo Bo|  Pass|
+-------+------+



In [3]:
df=spark.read.json(r"C:\vscode\data_mining\datasets\0.jsonl")
df.printSchema()

root
 |-- events: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- aid: long (nullable = true)
 |    |    |-- ts: long (nullable = true)
 |    |    |-- type: long (nullable = true)
 |-- session: long (nullable = true)



In [4]:
# Phẳng hóa events và trích xuất thông tin
df_flat = df.select(
    F.col("session"),
    F.explode(F.col("events")).alias("event")
).select(
    "session",
    F.col("event.aid").alias("aid"),
    F.col("event.ts").alias("ts"),
    F.col("event.type").alias("type")
)
df_flat.show(5, truncate=False)

+-------+---+-------------+----+
|session|aid|ts           |type|
+-------+---+-------------+----+
|0      |0  |1659304800025|0   |
|0      |1  |1659304904511|0   |
|0      |2  |1659367439426|0   |
|0      |3  |1659367719997|0   |
|0      |4  |1659367871344|0   |
+-------+---+-------------+----+
only showing top 5 rows



Tiền xử lí dữ liệu

In [5]:
# Thiết lập ngưỡng "Người thật"
MAX_APS = 2.0        # Không quá 2 hành động/giây
MIN_DURATION = 1.0   # Phiên phải dài ít nhất 1 giây nếu có nhiều hành động
MAX_DURATION = 86400 # Không quá 1 ngày
# Tính toán các chỉ số cho mỗi phiên
session_stats=df_flat.groupBy("session").agg(
    F.count("aid").alias("total_actions"),
    F.sum(F.when(F.col("type") == 0, 1).otherwise(0)).alias("total_clicks"),
    F.sum(F.when(F.col("type") == 1, 1).otherwise(0)).alias("total_carts"),
    F.sum(F.when(F.col("type") == 2, 1).otherwise(0)).alias("total_orders"),
    F.min("ts").alias("start_time"),
    F.from_unixtime(F.min("ts")/1000).alias("start_datetime"),
    F.max("ts").alias("end_time"),
    F.from_unixtime(F.max("ts")/1000).alias("end_datetime"),
    # Tính toán duration (Giây) = (Max - Min) / 1000
    ((F.max("ts") - F.min("ts")) / 1000).alias("duration_seconds")
)
session_stats=session_stats.withColumn("actions_per_second", F.col("total_actions") / (F.col("duration_seconds")+0.1))
session_stats.show(5, truncate=False)

+-------+-------------+------------+-----------+------------+-------------+-------------------+-------------+-------------------+----------------+---------------------+
|session|total_actions|total_clicks|total_carts|total_orders|start_time   |start_datetime     |end_time     |end_datetime       |duration_seconds|actions_per_second   |
+-------+-------------+------------+-----------+------------+-------------+-------------------+-------------+-------------------+----------------+---------------------+
|26     |43           |34          |6          |3           |1659304800411|2022-08-01 05:00:00|1660811509768|2022-08-18 15:31:49|1506709.357     |2.8539012481953245E-5|
|29     |76           |67          |9          |0           |1659304800483|2022-08-01 05:00:00|1661119746922|2022-08-22 05:09:06|1814946.439     |4.187451165469288E-5 |
|474    |125          |108         |17         |0           |1659304807691|2022-08-01 05:00:07|1661363586690|2022-08-25 00:53:06|2058778.999     |6.0715595

In [6]:
session_stats=session_stats.withColumn("is_human",
    (F.col("actions_per_second") <= MAX_APS) &
    (F.col("duration_seconds") >= MIN_DURATION) &
    (F.col("duration_seconds") <= MAX_DURATION) &
    (F.col("total_actions") >= 2) & 
    (F.col("total_actions") <= 250)  
)
print(f"Tong so hanh dong luc dau: {df_flat.count()}")
print(f"Tong so phien luc dau: {session_stats.count()}")
print(f"Tong so phien duoc xac dinh la nguoi that: {session_stats.filter(F.col('is_human') == True).count()}")
session_stats=session_stats.filter(F.col("is_human") == True)
session_stats.show(5, truncate=False)

Tong so hanh dong luc dau: 1127647
Tong so phien luc dau: 20000
Tong so phien duoc xac dinh la nguoi that: 3236
+-------+-------------+------------+-----------+------------+-------------+-------------------+-------------+-------------------+----------------+---------------------+--------+
|session|total_actions|total_clicks|total_carts|total_orders|start_time   |start_datetime     |end_time     |end_datetime       |duration_seconds|actions_per_second   |is_human|
+-------+-------------+------------+-----------+------------+-------------+-------------------+-------------+-------------------+----------------+---------------------+--------+
|964    |6            |6           |0          |0           |1659304815863|2022-08-01 05:00:15|1659310576551|2022-08-01 06:36:16|5760.688        |0.0010415241803725462|true    |
|293    |10           |7           |3          |0           |1659304804734|2022-08-01 05:00:04|1659305408507|2022-08-01 05:10:08|603.773         |0.01655977332982266  |true    

In [7]:
results=session_stats.filter(F.col("total_orders") > 0)
print(f"Tong so phien co it nhat 1 don hang: {results.count()}")
results.show(5, truncate=False)

Tong so phien co it nhat 1 don hang: 233
+-------+-------------+------------+-----------+------------+-------------+-------------------+-------------+-------------------+----------------+---------------------+--------+
|session|total_actions|total_clicks|total_carts|total_orders|start_time   |start_datetime     |end_time     |end_datetime       |duration_seconds|actions_per_second   |is_human|
+-------+-------------+------------+-----------+------------+-------------+-------------------+-------------+-------------------+----------------+---------------------+--------+
|299    |106          |67          |27         |12          |1659304804803|2022-08-01 05:00:04|1659388208843|2022-08-02 04:10:08|83404.04        |0.0012709201245885396|true    |
|237    |44           |33          |6          |5           |1659304803862|2022-08-01 05:00:03|1659373798796|2022-08-02 00:09:58|68994.934       |6.37727057283572E-4  |true    |
|463    |5            |2           |1          |2           |16593048

Xây dựng ma trận tương đồng

In [8]:
df_optimized=df_flat.join(session_stats.select("session", "is_human"), on="session", how="inner").filter(F.col("is_human") == True).drop("is_human")

In [9]:
# Mặc dù Spark xử lý tốt dữ liệu lớn, nhưng với OTTO, bạn nên ép kiểu (cast) 
# về các dạng nhỏ hơn để tiết kiệm RAM khi thực hiện lệnh join (bước tốn tài nguyên nhất).
from pyspark.sql import types as T
df_optimized = df_flat.select(
    F.col("session").cast(T.IntegerType()),
    F.col("aid").cast(T.IntegerType()),
    F.col("ts").cast(T.LongType()), # Timestamp nên để Long hoặc Integer tùy độ lớn
    F.col("type").cast(T.ByteType()) # Type chỉ có 0, 1, 2 nên dùng Byte (8-bit) là đủ
)

Gán trọng số để các hành động quan trọng hơn (mua hàng) có tiếng nói lớn hơn trong ma trận gợi ý.

Click (0): 1.0

Cart (1): 3.0

Order (2): 6.0

In [10]:
df_weighted = df_optimized.withColumn("weight", 
    F.when(F.col("type") == 0, 1.0)
     .when(F.col("type") == 1, 3.0)
     .otherwise(6.0)
)
df_weighted.show(5, truncate=False)

+-------+---+-------------+----+------+
|session|aid|ts           |type|weight|
+-------+---+-------------+----+------+
|0      |0  |1659304800025|0   |1.0   |
|0      |1  |1659304904511|0   |1.0   |
|0      |2  |1659367439426|0   |1.0   |
|0      |3  |1659367719997|0   |1.0   |
|0      |4  |1659367871344|0   |1.0   |
+-------+---+-------------+----+------+
only showing top 5 rows



Ma trận Co-visitation

In [11]:
# 1. Cấu hình các tham số lọc
MAX_TIME_GAP = 24 * 60 * 60 * 1000  # 24 giờ tính bằng miliseconds
MAX_CANDIDATES_PER_ITEM = 50 # Mỗi món đồ chỉ lưu 50 món liên quan nhất để tiết kiệm bộ nhớ

# 2. Thực hiện Self-Join
# Chúng ta dùng alias 'a' và 'b' để phân biệt hai phía của phép join
covis_matrix = df_weighted.alias("a").join(
    df_weighted.alias("b"),
    (F.col("a.session") == F.col("b.session")) & # Cùng một khách hàng
    (F.col("a.aid") != F.col("b.aid")) &           # Không tự kết nối chính nó
    (F.abs(F.col("a.ts") - F.col("b.ts")) < MAX_TIME_GAP) # Gần nhau về thời gian
)

# 3. Tính toán điểm số tương quan
covis_matrix = covis_matrix.groupBy("a.aid", "b.aid") \
    .agg(F.sum(F.col("b.weight")).alias("total_score"))

# 4. (Tùy chọn) Lọc bớt để ma trận gọn nhẹ
# Chỉ giữ lại những cặp có độ tương quan thực sự (xuất hiện > 1 lần)
covis_matrix = covis_matrix.filter(F.col("total_score") > 1)

# Hiển thị kết quả
covis_matrix.sort(F.desc("total_score")).show(10)
covis_matrix.count()

+------+------+-----------+
|   aid|   aid|total_score|
+------+------+-----------+
|  4929|  4922|     5683.0|
|  4922|  4929|     5069.0|
|131230| 12293|     2691.0|
| 12293|131230|     2691.0|
| 21288|112430|     2544.0|
|112430| 21288|     2433.0|
|121503|  4929|     2376.0|
| 30567| 23708|     1795.0|
|  4929|121503|     1768.0|
| 18164| 34607|     1501.0|
+------+------+-----------+
only showing top 10 rows



7050193